In [ ]:
# @title 1. Install Dependencies
# @markdown Run this cell first to install required libraries.
!pip install -q git+https://github.com/huggingface/transformers.git
!pip install -q accelerate gradio torch torchaudio scipy pillow
!pip install -q liquid-audio
print("Dependencies installed!")

In [ ]:
# @title 2. Model Logic & Backend
import gradio as gr
import torch
import gc
from transformers import (
    AutoModelForCausalLM,
    AutoModelForImageTextToText,
    AutoProcessor,
    AutoTokenizer,
    TextIteratorStreamer
)
from threading import Thread
import torchaudio
import os
import numpy as np
from liquid_audio import LFM2AudioModel, LFM2AudioProcessor, ChatState, LFMModality

# --- Global State ---
current_model = None
current_processor = None # Tokenizer or Processor
current_model_name = ""

MODELS = {
    "LFM2.5-1.2B-Instruct": "LiquidAI/LFM2.5-1.2B-Instruct",
    "LFM2.5-1.2B-JP": "LiquidAI/LFM2.5-1.2B-JP",
    "LFM2.5-VL-1.6B": "LiquidAI/LFM2.5-VL-1.6B",
    "LFM2.5-Audio-1.5B": "LiquidAI/LFM2.5-Audio-1.5B"
}

def free_memory():
    """Unloads model to free GPU memory."""
    global current_model, current_processor, current_model_name
    if current_model is not None:
        del current_model
        del current_processor
        current_model = None
        current_processor = None
        current_model_name = ""
        gc.collect()
        torch.cuda.empty_cache()
        print("Memory cleared.")

def load_model(model_key):
    """Loads a specific model if not already loaded."""
    global current_model, current_processor, current_model_name

    model_id = MODELS[model_key]

    if current_model_name == model_key:
        return current_model, current_processor

    print(f"Loading {model_key}...")
    free_memory()

    try:
        if "Audio" in model_key:
            current_processor = LFM2AudioProcessor.from_pretrained(model_id)
            current_model = LFM2AudioModel.from_pretrained(model_id).to("cuda")
        elif "VL" in model_key:
            current_processor = AutoProcessor.from_pretrained(model_id)
            current_model = AutoModelForImageTextToText.from_pretrained(
                model_id, device_map="auto", torch_dtype=torch.bfloat16
            )
        else: # Text models (Instruct / JP)
            current_processor = AutoTokenizer.from_pretrained(model_id)
            current_model = AutoModelForCausalLM.from_pretrained(
                model_id, device_map="auto", torch_dtype=torch.bfloat16
            )

        current_model_name = model_key
        print(f"{model_key} loaded successfully.")
        return current_model, current_processor
    except Exception as e:
        print(f"Error loading model: {e}")
        return None, None

def generate_text_instruct(model, tokenizer, prompt, history, system_prompt, **kwargs):
    """Handler for Text-only models."""
    try:
        messages = [{"role": "system", "content": system_prompt}] + history + [{"role": "user", "content": prompt}]

        inputs = tokenizer.apply_chat_template(
            messages, add_generation_prompt=True, return_tensors="pt", tokenize=True
        ).to(model.device)

        # Robust input extraction
        if hasattr(inputs, "input_ids"):
            input_ids = inputs.input_ids
        elif isinstance(inputs, dict) and "input_ids" in inputs:
            input_ids = inputs["input_ids"]
        else:
            input_ids = inputs

        streamer = TextIteratorStreamer(tokenizer, skip_prompt=True, skip_special_tokens=True)
        generation_kwargs = dict(
            input_ids=input_ids,
            streamer=streamer,
            **kwargs
        )

        thread = Thread(target=model.generate, kwargs=generation_kwargs)
        thread.start()

        partial_text = ""
        for new_text in streamer:
            partial_text += new_text
            yield partial_text
    except Exception as e:
        yield f"Error: {str(e)}"

def generate_vision(model, processor, prompt, image, history, system_prompt, **kwargs):
    """Handler for Vision-Language model."""
    if image is None:
        yield "Please upload an image for the VL model."
        return

    # --- FIX: Merge System Prompt into User Message ---
    # This avoids the "System role must be string vs Processor requires list" conflict.
    full_user_prompt = f"System Instruction: {system_prompt}\n\nUser Question: {prompt}"

    conversation = [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": image},
                {"type": "text", "text": full_user_prompt},
            ],
        },
    ]

    try:
        inputs = processor.apply_chat_template(
            conversation,
            add_generation_prompt=True,
            return_tensors="pt",
            return_dict=True,
            tokenize=True
        ).to(model.device)

        kwargs.pop("min_p", None)

        outputs = model.generate(**inputs, **kwargs)

        # Slicing logic
        input_len = inputs["input_ids"].shape[1]
        generated_tokens = outputs[:, input_len:]
        decoded = processor.batch_decode(generated_tokens, skip_special_tokens=True)[0]

        yield decoded
    except Exception as e:
        yield f"Error in VL generation: {str(e)}"

def generate_audio(model, processor, prompt, audio_path, history, system_prompt, **kwargs):
    """Handler for Audio model."""
    chat = ChatState(processor)

    # System Prompt
    full_system_prompt = f"{system_prompt} Respond with interleaved text and audio."

    chat.new_turn("system")
    chat.add_text(full_system_prompt)
    chat.end_turn()

    # User Input
    chat.new_turn("user")
    if audio_path:
        wav, sr = torchaudio.load(audio_path)
        chat.add_audio(wav, sr)
        if prompt:
             chat.add_text(prompt)
    else:
        chat.add_text(prompt)
    chat.end_turn()

    chat.new_turn("assistant")

    text_accum = ""
    audio_out = []

    audio_temp = kwargs.get('temperature', 1.0)
    audio_top_k = kwargs.get('top_k', 50)

    gen_kwargs = {
        "max_new_tokens": kwargs.get('max_new_tokens', 256),
        "audio_temperature": audio_temp,
        "audio_top_k": audio_top_k
    }

    try:
        for t in model.generate_interleaved(**chat, **gen_kwargs):
            if t.numel() == 1:
                chunk = processor.text.decode(t)
                chunk = chunk.replace("<|text_end|>", "")
                text_accum += chunk
                yield text_accum, None
            else:
                audio_out.append(t)

        out_path = None
        if audio_out:
            audio_codes = torch.stack(audio_out[:-1], 1).unsqueeze(0)
            waveform = processor.decode(audio_codes)
            out_path = "response.wav"
            torchaudio.save(out_path, waveform.cpu(), 24000)

        yield text_accum, out_path

    except Exception as e:
        yield f"Error in audio generation: {str(e)}", None

# --- Main Wrapper ---
def chat_wrapper(message, history, model_key, image_input, audio_input, system_prompt, temp, max_tok, top_p, min_p, rep_pen):
    model, processor = load_model(model_key)

    if not model:
        yield "Failed to load model.", None

    gen_kwargs = {
        "max_new_tokens": int(max_tok),
        "temperature": float(temp),
        "top_p": float(top_p),
        "repetition_penalty": float(rep_pen),
        "do_sample": True
    }

    if "VL" not in model_key and "Audio" not in model_key:
        gen_kwargs["min_p"] = float(min_p)
    else:
        gen_kwargs.pop("min_p", None)

    if "VL" in model_key:
        for response in generate_vision(model, processor, message, image_input, history, system_prompt, **gen_kwargs):
            yield response, None
    elif "Audio" in model_key:
        for text_resp, audio_resp in generate_audio(model, processor, message, audio_input, history, system_prompt, **gen_kwargs):
            yield text_resp, audio_resp
    else:
        for response in generate_text_instruct(model, processor, message, history, system_prompt, **gen_kwargs):
            yield response, None

In [ ]:
# @title 3. Launch Gradio Interface
with gr.Blocks(title="Liquid AI LFM2.5 Playground") as demo:
    gr.Markdown("# 💧 Liquid AI LFM2.5 Multi-Model Playground")
    gr.Markdown("Select a model to start. **Note:** Switching models takes a moment as it unloads/reloads GPU memory.")

    with gr.Row():
        with gr.Column(scale=1):
            model_selector = gr.Dropdown(
                choices=list(MODELS.keys()),
                value="LFM2.5-1.2B-Instruct",
                label="Select Model",
                interactive=True
            )

            # --- Dynamic Inputs ---
            image_box = gr.Image(type="pil", label="Image Input (For VL Model)", visible=False)
            audio_box = gr.Audio(type="filepath", label="Audio Input (For Audio Model)", visible=False)

            # --- Parameters ---
            with gr.Accordion("Generation Parameters", open=True):
                system_prompt = gr.Textbox(
                    value="You are a helpful assistant.",
                    label="System Prompt",
                    lines=2,
                    placeholder="Enter system instructions here..."
                )
                temp_slider = gr.Slider(0.1, 2.0, value=0.7, label="Temperature")
                max_tokens = gr.Slider(64, 2048, value=512, step=64, label="Max New Tokens")
                top_p_slider = gr.Slider(0.0, 1.0, value=0.9, label="Top P")
                min_p_slider = gr.Slider(0.0, 1.0, value=0.05, label="Min P (Text Only)")
                rep_pen_slider = gr.Slider(1.0, 2.0, value=1.05, label="Repetition Penalty")

        with gr.Column(scale=2):
            chatbot = gr.Chatbot(label="Conversation", height=500, type="messages")
            audio_output = gr.Audio(label="Audio Response (Audio Model Only)", autoplay=True, visible=False)
            msg_input = gr.Textbox(label="Type your message here...", placeholder="Enter text...", lines=2)
            submit_btn = gr.Button("Send", variant="primary")
            clear_btn = gr.Button("Clear Chat")

    # --- Visibility Logic ---
    def update_visibility(model_name):
        show_img = "VL" in model_name
        show_audio = "Audio" in model_name
        return {
            image_box: gr.update(visible=show_img, value=None),
            audio_box: gr.update(visible=show_audio, value=None),
            audio_output: gr.update(visible=show_audio, value=None)
        }

    model_selector.change(
        fn=update_visibility,
        inputs=model_selector,
        outputs=[image_box, audio_box, audio_output]
    )

    # --- Chat Logic ---
    def user_msg(user_message, history):
        return "", history + [{"role": "user", "content": user_message}]

    def bot_msg(history, model_key, img, aud, sys_prompt, temp, max_tok, tp, mp, rp):
        # Extract last user message
        user_message = history[-1]['content']

        # Add placeholder for assistant
        history.append({"role": "assistant", "content": ""})

        # History context (excluding current user msg and empty assistant placeholder)
        context = history[:-2]

        generator = chat_wrapper(user_message, context, model_key, img, aud, sys_prompt, temp, max_tok, tp, mp, rp)

        for text_resp, audio_resp in generator:
            history[-1]['content'] = text_resp
            yield history, audio_resp

    # --- Event Binding ---
    # Note: Added system_prompt to inputs list
    inputs_list = [chatbot, model_selector, image_box, audio_box, system_prompt, temp_slider, max_tokens, top_p_slider, min_p_slider, rep_pen_slider]

    submit_btn.click(
        user_msg, [msg_input, chatbot], [msg_input, chatbot], queue=False
    ).then(
        bot_msg, inputs_list, [chatbot, audio_output]
    )

    msg_input.submit(
        user_msg, [msg_input, chatbot], [msg_input, chatbot], queue=False
    ).then(
        bot_msg, inputs_list, [chatbot, audio_output]
    )

    clear_btn.click(lambda: [], None, chatbot, queue=False)

demo.queue().launch(share=True, debug=True)